# FFAI Agent Tools, JSON Repair, and Response Validation

This notebook demonstrates the new functionality ported from Plico:

1. **Fault-tolerant JSON parsing** -- `json_repair` handles malformed LLM JSON output
2. **`response_format` passthrough** -- request structured JSON from the provider
3. **Tool registry and execution** -- register Python callables as tools the LLM can invoke
4. **Agentic tool-call loop** -- `AgentLoop` executes multi-round tool-call cycles
5. **LLM-as-judge validation** -- `ResponseValidator` evaluates and retries responses

The model used is Mistral Small via LiteLLM.

---

## Setup

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import json  # noqa: E402
import os  # noqa: E402

from ffai.agent.agent_loop import AgentLoop  # noqa: E402
from ffai.agent.response_validator import ResponseValidator  # noqa: E402
from ffai.Clients import FFLiteLLMClient  # noqa: E402
from ffai.core.response_options import ResponseOptions  # noqa: E402
from ffai.core.response_utils import clean_response, extract_json  # noqa: E402
from ffai.FFAI import FFAI  # noqa: E402
from ffai.tools.tool_registry import ToolDefinition, ToolRegistry  # noqa: E402

api_key = os.getenv('MISTRAL_API_KEY')
if not api_key:
    raise RuntimeError('Set MISTRAL_API_KEY in your environment')

client = FFLiteLLMClient(
    model_string='mistral/mistral-small-latest',
    api_key=api_key,
    temperature=0.3,
    max_tokens=500,
)

ffai = FFAI(client)
print(f'FFAI initialized with model: {client.model}')

---

## 1. Fault-Tolerant JSON Parsing with `json_repair`

LLMs frequently produce slightly malformed JSON: trailing commas, unquoted keys,
single quotes instead of double quotes. FFAI now uses `json_repair` under the hood
in `extract_json()` and `clean_response()` to handle these cases.

In [ ]:
# Malformed JSON that json.loads would reject, but json_repair handles
malformed_samples = [
    '{"score": 8, "reason": "good",}',                          # trailing comma
    '{score: 5, items: [1, 2, 3,]}',                               # unquoted keys + trailing comma
    "{'name': 'Alice', 'age': 30}",                               # single quotes
]

for sample in malformed_samples:
    result = extract_json(sample)
    print(f'Input:    {sample}')
    print(f'Parsed:   {result}')
    print()

In [ ]:
# clean_response also handles JSON inside markdown code blocks
# (common when LLMs wrap output in ```json ... ```)
markdown_json = '```json\n{"analysis": "positive", "confidence": 0.92,}\n```'
result = clean_response(markdown_json)
print(f'Type:    {type(result).__name__}')
print(f'Content: {result}')

---

## 2. Structured JSON Output with `response_format`

FFAI now accepts a `response_format` parameter that passes through to the
provider API. Combined with `json_repair` in the response pipeline, this
gives you reliable structured output.

In [ ]:
# Request JSON output via response_format
# The prompt instructs the model what fields to include
result = ffai.generate_response(
    prompt=(
        'Analyze the sentiment of this text: '
        '"FFAI makes multi-provider AI orchestration much simpler."'
        '\n\nRespond with a JSON object with keys: sentiment (positive/negative/neutral), '
        'confidence (0-1), and explanation (one sentence).'
    ),
    prompt_name='sentiment',
    options=ResponseOptions(response_format={'type': 'json_object'}),
)

print(f'Type:     {type(result.response).__name__}')
print(f'Response: {json.dumps(result.response, indent=2) if isinstance(result.response, dict) else result.response}')
print(f'Tokens:   {result.usage}')
print(f'Cost:     ${result.cost_usd:.6f}')

---

## 3. Tool Registry: Registering Tools for LLM Use

The `ToolRegistry` lets you register Python functions as tools the LLM can invoke.
Each tool has a name, description, and JSON Schema for its parameters.
The registry produces OpenAI-format tool schemas that work with any provider.

In [ ]:
# Define Python functions that will be our tools

def calculate(args: dict) -> str:
    """Evaluate a simple arithmetic expression."""
    expr = args.get('expression', '')
    try:
        result = eval(expr, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f'Error: {e}'


def word_count(args: dict) -> str:
    """Count words in a given text."""
    text = args.get('text', '')
    count = len(text.split())
    return json.dumps({'word_count': count, 'text_length': len(text)})


# Register the tools
registry = ToolRegistry()

registry.register(ToolDefinition(
    name='calculate',
    description='Evaluate a simple arithmetic expression and return the result',
    parameters={
        'type': 'object',
        'properties': {
            'expression': {
                'type': 'string',
                'description': 'The arithmetic expression to evaluate (e.g. "2 + 3 * 4")',
            },
        },
        'required': ['expression'],
    },
))

registry.register(ToolDefinition(
    name='word_count',
    description='Count the number of words in a text string',
    parameters={
        'type': 'object',
        'properties': {
            'text': {
                'type': 'string',
                'description': 'The text to count words in',
            },
        },
        'required': ['text'],
    },
))

# Bind executors
registry.register_executor('calculate', calculate)
registry.register_executor('word_count', word_count)

print(f'Registered tools: {registry.get_registered_names()}')
print()

# Show the OpenAI-format schema that gets sent to the LLM
schemas = registry.get_tools_schema(['calculate', 'word_count'])
for schema in schemas:
    print(json.dumps(schema, indent=2))
    print()

In [ ]:
# Execute a tool directly (without LLM)
result = registry.execute_tool('calculate', {'expression': '17 * 23'})
print(f'calculate("17 * 23") = {result}')

result = registry.execute_tool('word_count', {'text': 'The quick brown fox jumps over the lazy dog'})
print(f'word_count result: {result}')

---

## 4. Agentic Tool-Call Loop

The `AgentLoop` sends a prompt to the LLM with the registered tools available.
If the LLM responds with tool calls, the loop executes each tool, feeds the results
back, and continues until the LLM produces a final answer without tool calls
or the maximum round count is reached.

In [ ]:
# Clone the client so the agent loop gets an isolated conversation
agent_client = client.clone()

agent_loop = AgentLoop(
    client=agent_client,
    tool_registry=registry,
    max_rounds=5,
    tool_timeout=10.0,
    continue_on_tool_error=True,
)

# Ask a question that requires both tools
agent_result = agent_loop.execute(
    prompt=(
        'Calculate 144 divided by 12, then count the words in this sentence: '
        '"The result of the calculation is interesting." '
        'Report both results.'
    ),
    tools=['calculate', 'word_count'],
)

print(f'Status:        {agent_result.status}')
print(f'Total rounds:  {agent_result.total_rounds}')
print(f'LLM calls:     {agent_result.total_llm_calls}')
print(f'Tool calls:    {agent_result.tool_calls_count}')
print()

for tc in agent_result.tool_calls:
    print(f'  Round {tc.round}: {tc.tool_name}({tc.arguments}) = {tc.result}')
    if tc.error:
        print(f'    Error: {tc.error}')
    print(f'    Duration: {tc.duration_ms:.0f}ms')

print()
print(f'Final response:\n{agent_result.response}')

In [ ]:
# AgentResult is serializable via to_dict() / from_dict()
result_dict = agent_result.to_dict()
print(json.dumps(result_dict, indent=2, default=str)[:1000])

---

## 5. LLM-as-Judge Response Validation

The `ResponseValidator` uses a second LLM call to evaluate whether a response
meets specified criteria. On failure, it can re-execute a function with the
rejection reason appended to the prompt, then re-validate the new response.

In [ ]:
# Validate a good response
validator = ResponseValidator(client, temperature=0.1)

good_response = json.dumps({
    'sentiment': 'positive',
    'confidence': 0.95,
    'explanation': 'The text expresses enthusiasm about FFAI.',
})

result = validator.validate(
    response=good_response,
    criteria='The response must be valid JSON with keys: sentiment, confidence, explanation',
)

print(f'Passed:   {result.passed}')
print(f'Attempts: {result.attempts}')
print(f'Critique: {result.critique}')

In [ ]:
# Validate a bad response with re-execution on failure
# The re_execute_fn generates an improved response when validation fails

call_count = {'count': 0}

def re_execute_fn(prompt: str):
    """Simulate re-execution: produce a better response."""
    call_count['count'] += 1
    improved = ffai.generate_response(
        prompt=prompt,
        prompt_name='improved_response',
        options=ResponseOptions(response_format={'type': 'json_object'}),
    )
    return improved

bad_response = 'I think the sentiment is probably positive.'

result = validator.validate(
    response=bad_response,
    criteria=(
        'The response must be a JSON object (not plain text) with exactly these keys: '
        'sentiment (string), confidence (number between 0-1), explanation (string)'
    ),
    max_retries=2,
    re_execute_fn=re_execute_fn,
)

print(f'Passed:         {result.passed}')
print(f'Attempts:       {result.attempts}')
print(f'Re-executions:  {call_count["count"]}')
print(f'Critique:       {result.critique}')

---

## 6. Putting It All Together: Structured Agent Pipeline

A realistic workflow combining all features:
1. Request structured JSON output (`response_format`)
2. Parse with fault-tolerant `json_repair`
3. Use `AgentLoop` with tools for computation
4. Validate the final result with `ResponseValidator`

In [ ]:
# Step 1: Use AgentLoop to answer a question that needs tools
pipeline_client = client.clone()

pipeline_loop = AgentLoop(
    client=pipeline_client,
    tool_registry=registry,
    max_rounds=4,
)

agent_result = pipeline_loop.execute(
    prompt=(
        'Calculate the product of 13 and 7, then count the words in: '
        '"Thirteen times seven equals something." '
        'Finally, summarize both results in one sentence.'
    ),
    tools=['calculate', 'word_count'],
)

print('=== Agent Result ===')
print(f'Status: {agent_result.status}')
print(f'Rounds: {agent_result.total_rounds}')
print(f'Tools:  {agent_result.tool_calls_count}')
for tc in agent_result.tool_calls:
    print(f'  {tc.tool_name}({tc.arguments}) => {tc.result}')
print()

# Step 2: Validate the agent's final response
validation = validator.validate(
    response=agent_result.response,
    criteria='The response must mention both the calculation result and the word count result.',
    max_retries=1,
)

print('=== Validation ===')
print(f'Passed:   {validation.passed}')
print(f'Attempts: {validation.attempts}')
if validation.critique:
    print(f'Critique: {validation.critique}')
print()

print('=== Final Response ===')
print(agent_result.response)

---

## Session Cost Summary

In [ ]:
print(ffai.get_model_usage_stats())
print()

stats_df = ffai.get_prompt_name_stats_df()
print(stats_df)